# PEFTGuard retrained on the shared development split (Sandra)

Use a reviewed pinned source commit and the custodian's verified development bundle. Fit on train, early-stop on validation, freeze, then calibrate on independent benign records. The optional pretrained reference is separate. Evaluation and final assessment remain with the custodian.

## 1. GPU, dependencies, memory plan

In [ ]:
!pip -q install safetensors
import torch
gpu = torch.cuda.get_device_properties(0)
total_gb = gpu.total_memory / 2**30
print(gpu.name, f"{total_gb:.0f} GB")
assert total_gb >= 26, "PEFTGuard Llama-2-7B needs a >=26 GB GPU (A100-40GB recommended); an L4 (24 GB) does not fit. Runtime > Change runtime type"

## 2. Configuration
Supply SOURCE_COMMIT, BUNDLE_SHA256 and BINDING_SHA256 from the trusted handoff receipt. Full manifests and split plans stay with the custodian. ADAPTER_ROOT contains only staged development files under neutral IDs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL   = "https://github.com/JuliaIssa2/FYP-AI-Antivirus.git"   # <- the team repo
SOURCE_COMMIT = ""  # full reviewed 40-character commit supplied by the custodian
BUNDLE = "/content/drive/MyDrive/fyp_shared/development.json"
BUNDLE_SHA256 = ""   # trusted file checksum, supplied independently
BINDING_SHA256 = ""  # trusted ProtocolBinding fingerprint
ADAPTER_ROOT = "/content/adapters"                              # <root>/<adapter_id>/adapter_model.safetensors
RUN_DIR    = "/content/runs/peftguard_seed0"                    # local disk, written once, never overwritten
DRIVE_OUT  = "/content/drive/MyDrive/fyp_shared/peftguard_seed0"
SEED = 0
PLAN = "auto"   # auto | adafactor_fp32 | adam_bf16 | adam_fp32  (auto picks by GPU memory, see the next section)

## 3. Get the code

In [ ]:
import os, sys, shutil, re, subprocess
from pathlib import Path

if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
    raise ValueError("Supply the full reviewed source commit from the custodian")
repo = Path("/content/repo")
if not repo.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(repo)], check=True)
elif subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Use a clean reviewed checkout")
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", SOURCE_COMMIT], check=True)
if subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError("Source commit mismatch")
if any(k == "shared_protocol" or k.startswith("shared_protocol.") for k in sys.modules):
    raise RuntimeError("Restart the runtime before loading the reviewed source")
os.chdir(repo)
sys.path.insert(0, str(repo))
print("Reviewed source:", SOURCE_COMMIT)

## 4. Validate the complete development bundle

In [ ]:
from shared_protocol.development import load_development_bundle

development = load_development_bundle(
    BUNDLE, expected_file_sha256=BUNDLE_SHA256, expected_binding_sha256=BINDING_SHA256)
binding = development.binding
train_records = development.records("train")
val_records = development.records("validation")
cal_records = development.records("calibration")
print({p: len(development.records(p)) for p in ("train", "validation", "calibration")})

## 5. Verify custodian-staged adapter files
Files must be at ADAPTER_ROOT/<adapter_id>/adapter_model.safetensors. Metadata validation completes before any checkpoint is opened. All supplied development files, including existing files, must match their expected hashes. No Hugging Face credentials or adapter downloads are needed.

In [ ]:
from shared_protocol.development import verify_development_checkpoints

# Custodian-staged files only. Existing files are always hashed; no downloads.
resolver = verify_development_checkpoints(
    development, ADAPTER_ROOT, expected_binding_sha256=BINDING_SHA256)
print("Verified all staged development checkpoints")

## 6. Train (retrain PEFTGuard on exactly the shared train split)
Memory plans, MEASURED on an A100-40GB (peak GPU memory): `adam_fp32` (the authors' recipe) runs out of memory and needs a >=70 GB GPU;
`adafactor_fp32` (fp32 weights, Adafactor) peaks at 28.1 GB; `adam_bf16` (everything bf16) at 22.0 GB. 8-bit Adam is NOT used: it crashes on the
2^31-element `fc1` weight. `auto` picks `adafactor_fp32` on a 40 GB A100. Any plan other than `adam_fp32` is a deviation: state it with the results.
The plan is written into `fit.json`. Run a 1-epoch smoke test first (`EPOCHS = 1`, a different `RUN_DIR`) and check the loss is finite and falling.
For a cross-check run, set `PLAN = "adam_bf16"` with its own `RUN_DIR`/`DRIVE_OUT`.

In [ ]:
from shared_protocol.detectors.peftguard.train import TrainConfig, PLANS, auto_memory_plan, fit
from shared_protocol.detectors.peftguard.converter import root_resolver

EPOCHS = 20
optimizer, param_dtype = auto_memory_plan(total_gb) if PLAN == "auto" else PLANS[PLAN]
cfg = TrainConfig(epochs=EPOCHS, seed=SEED, optimizer=optimizer, param_dtype=param_dtype)
print(cfg)
record = fit(binding, train_records, val_records, cfg, RUN_DIR, resolver=root_resolver(ADAPTER_ROOT, restricted=True))
print("checkpoint sha256:", record["checkpoint_sha256"])

## 7. Keep the run (Drive). Write the checkpoint SHA256 into the team log: it is the detector's frozen state.

In [ ]:
os.makedirs(DRIVE_OUT, exist_ok=False)
for name in ("peftguard_best.pt", "fit.json"):
    shutil.copyfile(f"{RUN_DIR}/{name}", f"{DRIVE_OUT}/{name}")
print(open(f"{DRIVE_OUT}/fit.json").read()[:300], "...")

## 8. Freeze: clean-only calibration threshold (5% target)
Uses only the calibration partition (one representative per leakage group). No evaluation data is touched.

In [ ]:
from shared_protocol.detectors.peftguard.detector import PEFTGuardDetector
from shared_protocol.scoring import score_development
from shared_protocol.calibration import calibrate_benign, save_threshold

detector = PEFTGuardDetector.from_fit(f"{RUN_DIR}/fit.json", f"{RUN_DIR}/peftguard_best.pt", binding,
                                      expected_checkpoint_sha256=record["checkpoint_sha256"],
                                      resolver=root_resolver(ADAPTER_ROOT, restricted=True))
cal_scores = score_development(detector, cal_records, binding, "calibration")
threshold = calibrate_benign(binding, cal_records, cal_scores)
print("n calibration:", len(cal_scores.scores), "| threshold:", threshold.value)
import statistics
raws = sorted(s.raw for s in cal_scores.scores)
print("raw logit-difference range:", raws[0], "to", raws[-1], "| distinct:", len(set(raws)))
save_threshold(f"{DRIVE_OUT}/threshold.json", threshold, binding)

## 9. (Optional) Reference line: the authors' pretrained Llama-2-7B checkpoint
Reference only. Its training adapters are unknown (question 1 to Zhen Sun), so it is never the baseline. Needs ~13 GB + 8.6 GB download.

In [ ]:
RUN_PRETRAINED_REFERENCE = False  # separate optional experiment; skipped by default
if RUN_PRETRAINED_REFERENCE:
    from huggingface_hub import hf_hub_download
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
    from shared_protocol.detectors.peftguard.converter import sha256_file
    CKPT_FILE = "toxic-backdoors-hard/llama2_7b_toxic_backdoors_hard1/best_model.pth"
    ckpt = hf_hub_download("Vincent-HKUSTGZ/PEFTGuard_For_Llama2_7B", CKPT_FILE, token=HF_TOKEN)
    ckpt_sha = sha256_file(ckpt); print("pretrained sha256:", ckpt_sha)   # record this in the team log
    ref = PEFTGuardDetector.pretrained_reference(
        ckpt, expected_sha256=ckpt_sha, resolver=root_resolver(ADAPTER_ROOT, restricted=True),
        provenance="Authors' released checkpoint " + CKPT_FILE + "; training adapters unknown (author not yet answered)")
    ref_cal = score_development(ref, cal_records, binding, "calibration")
    print("reference calibration threshold:", calibrate_benign(binding, cal_records, ref_cal).value)

## 10. Custodian evaluation only

Development ends after model and threshold freeze. Send the frozen artifacts, hashes, source commit and run configuration to the custodian. This notebook cannot load evaluation or final-holdout records. The custodian uses the separate shared evaluation/release workflow after all detectors are frozen.